# U4A3 — Modelo Introductorio de Clasificación
## Gemelo digital de robot seguidor de línea (CPS + IA)

**Proyecto:** Clasificación del estado de operación del robot seguidor de línea
**Dataset:** `datos_limpios_v0.2.csv` (versión v0.2, capturada el 03/06/2026)
**Tipo de problema:** Clasificación multiclase

---

### Descripción de las variables utilizadas
- **Entradas (features):** `s0_cal`, `s1_cal`, `s2_cal`, `s3_cal` — lecturas calibradas (0–1000) de los 4 sensores QTR-1A.
- **Variable objetivo (target):** `estado` — estado del robot: `centrado`, `desviado_izq`, `desviado_der`, `recuperacion`.

### Justificación de las variables
Se eligen las **lecturas de los sensores** como entrada porque son los datos primarios del mundo real
que percibe el robot. Esto permite que el modelo aprenda a inferir el estado a partir de la percepción,
tal como lo haría el gemelo digital. **Se evita deliberadamente** usar `error_qtr` o `posicion_qtr` como
entrada, ya que el estado se deriva directamente de ellos en el firmware del robot; usarlos produciría una
fuga de información (data leakage) y un accuracy artificialmente perfecto que no demostraría capacidad real
de aprendizaje.

In [ ]:
# === 1. Importar librerías ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEMILLA = 42   # semilla fija para reproducibilidad
np.random.seed(SEMILLA)
print("Librerías cargadas. Semilla fija =", SEMILLA)

## 2. Carga de datos

In [ ]:
# === 2. Cargar el dataset v0.2 ===
df = pd.read_csv("datos_limpios_v0.2.csv")
print("Dataset cargado:", df.shape[0], "filas,", df.shape[1], "columnas")
df.head()

In [ ]:
# Distribución de clases del target
print("Distribución del estado del robot:")
print(df['estado'].value_counts())
print("\nPorcentajes:")
print((df['estado'].value_counts(normalize=True)*100).round(1))

## 3. Selección de variables y partición de datos

Usamos los 4 sensores como entrada y el estado como objetivo. Separamos en
**75% entrenamiento / 25% prueba**, con semilla fija (42) y estratificación
(para mantener la proporción de clases en ambos conjuntos).

In [ ]:
# === 3. Selección de variables ===
features = ['s0_cal', 's1_cal', 's2_cal', 's3_cal']
X = df[features]
y = df['estado']

# Partición train/test (estratificada, semilla fija)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)
print(f"Entrenamiento: {len(X_train)} muestras")
print(f"Prueba:        {len(X_test)} muestras")

## 4. Línea base (baseline)

Para clasificación, el baseline predice siempre la **clase mayoritaria** (`centrado`).
Es la referencia mínima: cualquier modelo útil debe superarla.

In [ ]:
# === 4. Baseline: clase mayoritaria ===
baseline = DummyClassifier(strategy='most_frequent', random_state=SEMILLA)
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)

acc_base = accuracy_score(y_test, pred_base)
f1_base = f1_score(y_test, pred_base, average='macro', zero_division=0)
print(f"Baseline -> Accuracy: {acc_base:.3f} | F1-macro: {f1_base:.3f}")

## 5. Entrenamiento de modelos

Entrenamos tres modelos sencillos y los comparamos: **Árbol de decisión**, **KNN** y **Random Forest**.
La nota de la actividad recomienda no elegir el más complejo sin verificar mejora real, por eso comparamos
todos contra el baseline.

In [ ]:
# === 5. Entrenar varios modelos ===
modelos = {
    'Árbol de decisión': DecisionTreeClassifier(max_depth=6, random_state=SEMILLA),
    'KNN (k=5)':         KNeighborsClassifier(n_neighbors=5),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=SEMILLA),
}

resultados = {'Baseline (clase mayoritaria)': (acc_base, f1_base)}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average='macro', zero_division=0)
    resultados[nombre] = (acc, f1)
    print(f"{nombre:<22} -> Accuracy: {acc:.3f} | F1-macro: {f1:.3f}")

## 6. Métricas de desempeño

Usamos **Accuracy** (porcentaje de aciertos global) y **F1-score macro** (promedio del F1 por clase).
El F1-macro es especialmente adecuado aquí porque el dataset está **desbalanceado**: el accuracy por sí
solo puede engañar (un modelo que solo acierte la clase mayoritaria tendría accuracy decente pero F1-macro
bajo). El F1-macro penaliza el mal desempeño en las clases minoritarias.

In [ ]:
# === 6. Tabla comparativa de métricas ===
tabla = pd.DataFrame(
    [(k, round(v[0],3), round(v[1],3)) for k,v in resultados.items()],
    columns=['Modelo', 'Accuracy', 'F1-macro']
)
tabla['Mejora Acc. vs baseline'] = (tabla['Accuracy'] - acc_base).round(3)
print(tabla.to_string(index=False))

## 7. Selección del mejor modelo y matriz de confusión

In [ ]:
# === 7. Elegir el mejor modelo por F1-macro ===
mejor_nombre = max(
    [k for k in resultados if k != 'Baseline (clase mayoritaria)'],
    key=lambda k: resultados[k][1]
)
print("Mejor modelo:", mejor_nombre)

mejor_modelo = modelos[mejor_nombre]
pred_mejor = mejor_modelo.predict(X_test)

# Reporte por clase
print("\nReporte de clasificación del mejor modelo:")
print(classification_report(y_test, pred_mejor, zero_division=0))

In [ ]:
# === Matriz de confusión ===
clases = sorted(y.unique())
cm = confusion_matrix(y_test, pred_mejor, labels=clases)
fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
disp.plot(ax=ax, cmap='Greys', colorbar=False)
plt.title(f"Matriz de confusión - {mejor_nombre}")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("matriz_confusion.png", dpi=120, bbox_inches='tight')
plt.show()
print("Matriz de confusión guardada como matriz_confusion.png")

## 8. Importancia de las variables

In [ ]:
# === 8. ¿Qué sensores influyen más? (con Random Forest) ===
rf = RandomForestClassifier(n_estimators=100, random_state=SEMILLA)
rf.fit(X_train, y_train)
importancias = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Importancia de cada sensor:")
print(importancias.round(3))

fig, ax = plt.subplots(figsize=(6,3.5))
importancias.plot(kind='barh', ax=ax, color='#404040')
ax.set_title("Importancia de los sensores")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("importancia_variables.png", dpi=120, bbox_inches='tight')
plt.show()

## 9. Interpretación técnica

**¿El modelo mejora al baseline?**
Sí, ampliamente. El baseline alcanza ~58% de accuracy (y F1-macro de solo ~0.18, porque solo acierta una
clase). Los modelos entrenados superan el 92% de accuracy y ~0.92 de F1-macro. La mejora es clara y medible:
más de 35 puntos porcentuales de accuracy.

**¿Qué variables influyen más?**
Los sensores de los extremos (`s3_cal` y `s1_cal`) son los más informativos para distinguir el estado, lo
cual tiene sentido físico: las desviaciones se detectan principalmente en los bordes del arreglo de sensores.

**¿El desempeño es suficiente para integrar al gemelo digital?**
Sí. Un accuracy >92% y un F1-macro >0.92 indican que el modelo clasifica de forma confiable el estado del
robot a partir de los sensores, lo que es suficiente para una primera versión del gemelo digital y para
sustentar la comparación entre el robot real y el simulado.

**¿Qué limitaciones tiene el modelo o el dataset?**
- Las clases `desviado_izq` (43) y `recuperacion` (22) tienen pocas muestras, por lo que el modelo es menos
  confiable en ellas (visible en la matriz de confusión).
- El dataset proviene de una pista pequeña y recorridos cortos; podría no generalizar a pistas más complejas.
- Se recomienda ampliar el dataset (versión v0.3) con más muestras de las clases minoritarias antes de
  llevar el modelo a producción en el gemelo digital.

In [ ]:
# === 10. Guardar la tabla de métricas a CSV (trazabilidad) ===
tabla.to_csv("metricas_u4a3.csv", index=False)
print("Tabla de métricas guardada como metricas_u4a3.csv")
print("\n=== RESUMEN FINAL ===")
print(tabla.to_string(index=False))